In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split,StratifiedKFold,GridSearchCV,RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.compose import ColumnTransformer,make_column_selector
from sklearn.pipeline import Pipeline

from sklearn.metrics import roc_auc_score,roc_curve

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier , BaggingClassifier , AdaBoostClassifier , GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier , RadiusNeighborsClassifier
from sklearn.svm import SVC,NuSVC,LinearSVC
from sklearn.dummy import DummyClassifier

seed=1

In [13]:
df=pd.read_csv("/content/train.csv")
df

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
593989,593989,23004.26,0.152,703,20958.37,10.92,Female,Single,High School,Employed,Business,C3,1.0
593990,593990,35289.43,0.105,559,3257.24,14.62,Male,Single,Bachelor's,Employed,Debt consolidation,F5,1.0
593991,593991,47112.64,0.072,675,929.27,14.13,Female,Married,Bachelor's,Employed,Debt consolidation,C1,1.0
593992,593992,76748.44,0.067,740,16290.40,9.87,Male,Single,Bachelor's,Employed,Debt consolidation,B2,1.0


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 593994 entries, 0 to 593993
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype   
---  ------                --------------   -----   
 0   id                    593994 non-null  int64   
 1   annual_income         593994 non-null  float64 
 2   debt_to_income_ratio  593994 non-null  float64 
 3   credit_score          593994 non-null  int64   
 4   loan_amount           593994 non-null  float64 
 5   interest_rate         593994 non-null  float64 
 6   gender                593994 non-null  category
 7   marital_status        593994 non-null  category
 8   education_level       593994 non-null  category
 9   employment_status     593994 non-null  category
 10  loan_purpose          593994 non-null  category
 11  grade_subgrade        593994 non-null  category
 12  loan_paid_back        593994 non-null  bool    
dtypes: bool(1), category(6), float64(4), int64(2)
memory usage: 31.2 MB


In [7]:
df.columns

Index(['id', 'annual_income', 'debt_to_income_ratio', 'credit_score',
       'loan_amount', 'interest_rate', 'gender', 'marital_status',
       'education_level', 'employment_status', 'loan_purpose',
       'grade_subgrade', 'loan_paid_back'],
      dtype='object')

In [15]:
df[['gender', 'marital_status','education_level', 'employment_status', 'loan_purpose','grade_subgrade']]=df[['gender', 'marital_status','education_level', 'employment_status', 'loan_purpose','grade_subgrade']].astype('category')
df['loan_paid_back']=df['loan_paid_back'].astype(bool)

In [24]:
X_train,X_test,y_train,y_test=train_test_split(df.drop(['id','loan_paid_back'],axis=1),df['loan_paid_back'],test_size=0.2,random_state=seed,stratify=df['loan_paid_back'])


In [ ]:
cv_strat=StratifiedKFold(n_splits=5,shuffle=True,random_state=seed)

In [ ]:
model_config={
    "Dummy":{
        'model':DummyClassifier(strategy='most_frequent'),
        'params':{}
    },

    'LR':{
        'model':LogisticRegression(max_iter=2000,random_state=seed),
        'params':[
            {
                'c__solver':['saga','liblinear'],
                'c__penalty':['l1','l2'],
                'c__C':[0.1,1,10]
            },
            {
                'c__solver':['saga'],
                'c__penalty':['elasticnet'],
                'c__l1_ratio':[0.25,0.5,0.75],
                'c__C':[0.1,1,10]
            },
            {
                'c__solver':['lbfgs','newton-cg'],
                'c__penalty':['l2'],
                'c__C':[0.1,1,10]
            }

        ]
    },
    'DT':{
        'model':DecisionTreeClassifier(random_state=seed),
        'params':{
            'c__max_depth':[5,10,20,None],
            'c_class_weight':['balanced',None]
        }
    }
}